In [1]:
import argparse, json, os, sys, time, warnings
from datetime import date, timedelta
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             matthews_corrcoef, roc_auc_score, confusion_matrix)
from sklearn.base import clone
from xgboost import XGBClassifier

In [2]:
warnings.filterwarnings("ignore")
SEED = 42

UNIVERSE = {
    # equities (US large caps, diverse sectors)
    "AAPL": "Equity", "MSFT": "Equity", "NVDA": "Equity", "AMZN": "Equity",
    "JPM": "Equity", "XOM": "Equity", "JNJ": "Equity", "TSLA": "Equity",
    # equity-index ETFs / indices
    "SPY": "Index/ETF", "QQQ": "Index/ETF", "IWM": "Index/ETF", "EEM": "Index/ETF",
    "^GDAXI": "Index/ETF", "XU100.IS": "Index/ETF",
    # bonds, commodities, FX, crypto
    "TLT": "Bond", "GLD": "Commodity", "USO": "Commodity",
    "EURUSD=X": "FX", "BTC-USD": "Crypto", "ETH-USD": "Crypto",
}


In [3]:
def download(tickers, start, end, cache):
    import yfinance as yf
    out = {}
    for t in tickers:
        df = yf.download(t, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df[["Open", "High", "Low", "Close", "Volume"]].dropna(subset=["Close"])
        df.index = pd.to_datetime(df.index).tz_localize(None)
        out[t] = df
        df.to_csv(os.path.join(cache, f"{t.replace('^','').replace('=','_')}.csv"))
        print(f"  {t:10s} {len(df):5d} rows  {df.index[0].date()} -> {df.index[-1].date()}")
    return out

In [4]:
def load_dir(path, tickers):
    out = {}
    for t in tickers:
        for name in (t, t.replace("^", "").replace("=", "_")):
            f = os.path.join(path, f"{name}.csv")
            if os.path.exists(f):
                df = pd.read_csv(f, index_col=0, parse_dates=True)
                df.columns = [c.title() for c in df.columns]
                out[t] = df[["Open", "High", "Low", "Close", "Volume"]].dropna(subset=["Close"])
                break
    return out

In [5]:
def synthetic(tickers, n=780, seed=0):
    """Random-walk OHLCV with volatility clustering and NO predictability."""
    rng = np.random.default_rng(seed)
    idx = pd.bdate_range("2023-10-12", periods=n)
    out = {}
    for t in tickers:
        h = np.zeros(n); h[0] = 1e-4
        r = np.zeros(n)
        for i in range(1, n):
            h[i] = 2e-6 + 0.08 * r[i-1] ** 2 + 0.9 * h[i-1]
            r[i] = np.sqrt(h[i]) * rng.standard_normal() + 3e-4
        c = 100 * np.exp(np.cumsum(r))
        o = np.r_[c[0], c[:-1]] * np.exp(rng.normal(0, .003, n))
        hi = np.maximum(o, c) * (1 + np.abs(rng.normal(0, .004, n)))
        lo = np.minimum(o, c) * (1 - np.abs(rng.normal(0, .004, n)))
        v = rng.lognormal(15, .3, n) * (1 + 5 * np.abs(r))
        out[t] = pd.DataFrame(dict(Open=o, High=hi, Low=lo, Close=c, Volume=v), index=idx)
    return out

In [6]:
FEATURES = None
def make_features(df, leak_control=False):
    """Return (X, y, r). Row t: features known at close of t-1, target = sign of r_t."""
    o, h, l, c, v = (df[k].astype(float) for k in ["Open", "High", "Low", "Close", "Volume"])
    r = np.log(c / c.shift(1))                       # r_t
    F = pd.DataFrame(index=df.index)
    for k in (1, 2, 3, 5):
        F[f"ret_l{k}"] = r.shift(k - 1)             # r_t (as seen at day t) lagged k-1; after global shift -> r_{t-k}
    for w in (5, 10, 20):
        F[f"retmean_{w}"] = r.rolling(w).mean()
    for w in (5, 10, 20):
        F[f"vol_{w}"] = r.rolling(w).std()
    F["volratio_5_20"] = F["vol_5"] / F["vol_20"]
    F["volratio_10_20"] = F["vol_10"] / F["vol_20"]
    # RSI (Wilder, 14) scaled to [-.5,.5]
    d = c.diff(); up = d.clip(lower=0); dn = -d.clip(upper=0)
    rs = up.ewm(alpha=1/14, adjust=False).mean() / dn.ewm(alpha=1/14, adjust=False).mean().replace(0, np.nan)
    F["rsi14"] = (100 - 100 / (1 + rs)) / 100 - 0.5
    # MACD histogram normalised by price
    macd = c.ewm(span=12, adjust=False).mean() - c.ewm(span=26, adjust=False).mean()
    F["macd_hist"] = (macd - macd.ewm(span=9, adjust=False).mean()) / c
    # Bollinger %B and distance to SMAs
    m20, s20 = c.rolling(20).mean(), c.rolling(20).std()
    F["bb_pctb"] = (c - (m20 - 2 * s20)) / (4 * s20) - 0.5
    F["dist_sma20"] = c / m20 - 1
    F["dist_sma50"] = c / c.rolling(50).mean() - 1
    # stochastic %K, ATR/price
    ll, hh = l.rolling(14).min(), h.rolling(14).max()
    F["stoch_k"] = (c - ll) / (hh - ll).replace(0, np.nan) - 0.5
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    F["atr14_pct"] = tr.rolling(14).mean() / c
    # candle geometry (scale-free)
    F["hl_range"] = (h - l) / c
    F["oc_body"] = (c - o) / o
    F["upper_shadow"] = (h - np.maximum(o, c)) / c
    F["lower_shadow"] = (np.minimum(o, c) - l) / c
    F["gap"] = np.log(o / c.shift(1))
    # volume (relative / differenced only)
    lv = np.log(v.replace(0, np.nan))
    F["dlogvol"] = lv.diff()
    F["relvol_20"] = lv - np.log(v.rolling(20).mean())
    # --- timing: everything above is info at close of day t; shift by ONE day so row t uses t-1
    F = F.shift(1)
    # ret_l{k}: r.shift(k-1) then global shift(1)  => r_{t-k}.  (k=1 -> r_{t-1})
    y = (r > 0).astype(int)
    if leak_control:                                 # deliberately leaky control experiment
        F["LEAK_r_t"] = r
    ok = F.notna().all(axis=1) & r.notna()
    return F[ok], y[ok], r[ok]

In [7]:
def model_zoo():
    """(default estimator, search space) per model.  Inner CV is chronological."""
    zoo = {
        "KNN": (Pipeline([("sc", StandardScaler()), ("m", KNeighborsClassifier())]),
                {"m__n_neighbors": [5, 11, 21, 41, 81], "m__weights": ["uniform", "distance"],
                 "m__p": [1, 2]}),
        "RF": (RandomForestClassifier(random_state=SEED, n_jobs=1),
               {"n_estimators": [100, 200, 400], "max_depth": [2, 3, 5, 8, None],
                "min_samples_leaf": [1, 5, 10, 20], "max_features": ["sqrt", 0.5, 1.0]}),
        "GBM": (XGBClassifier(random_state=SEED, n_jobs=1, verbosity=0, eval_metric="logloss"),
                {"n_estimators": [50, 100, 200, 400], "max_depth": [1, 2, 3, 4],
                 "learning_rate": [0.005, 0.01, 0.03, 0.1], "subsample": [0.6, 0.8, 1.0],
                 "colsample_bytree": [0.6, 0.8, 1.0], "min_child_weight": [1, 5, 10],
                 "reg_lambda": [1, 10, 50]}),
        "ANN": (Pipeline([("sc", StandardScaler()),
                          ("m", MLPClassifier(random_state=SEED, max_iter=300,
                                              early_stopping=False))]),
                {"m__hidden_layer_sizes": [(16,), (32,), (64,), (32, 16), (64, 32)],
                 "m__activation": ["relu", "tanh"], "m__alpha": [1e-4, 1e-2, 1e-1, 1.0],
                 "m__learning_rate_init": [1e-3, 3e-3, 1e-2]}),
    }
    return zoo

In [8]:
def tune(est, space, X, y, n_iter, inner_splits, gap):
    cv = TimeSeriesSplit(n_splits=inner_splits, gap=gap)
    rs = RandomizedSearchCV(est, space, n_iter=n_iter, cv=cv, scoring="balanced_accuracy",
                            random_state=SEED, n_jobs=1, refit=True, error_score=0.0)
    rs.fit(X, y)
    return rs.best_estimator_, rs.best_params_, rs.best_score_

In [9]:
def metrics(y, p, prob):
    out = dict(acc=accuracy_score(y, p), bacc=balanced_accuracy_score(y, p),
               f1=f1_score(y, p, zero_division=0), mcc=matthews_corrcoef(y, p) if len(set(p)) > 1 else 0.0)
    try: out["auc"] = roc_auc_score(y, prob)
    except Exception: out["auc"] = np.nan
    return out

def walk_forward(X, y, r, init_train, test_len, gap, n_iter, inner_splits, models, log=None):
    """Expanding window.  Train = [0, a), purge `gap` rows, test = [a+gap, a+gap+test_len)."""
    n = len(X); zoo = model_zoo(); preds = []; chosen = []
    a = init_train; fold = 0
    while a + gap < n:
        te = np.arange(a + gap, min(a + gap + test_len, n))
        tr = np.arange(0, a)
        Xtr, ytr = X.iloc[tr], y.iloc[tr]; Xte, yte = X.iloc[te], y.iloc[te]
        base = pd.DataFrame(dict(date=X.index[te], y=yte.values, r=r.iloc[te].values, fold=fold))
        # reference rules
        base["pred_AlwaysUp"] = 1
        base["pred_Majority"] = int(ytr.mean() >= .5)
        base["pred_Persist"] = (X["ret_l1"].iloc[te].values > 0).astype(int)   # sign(r_{t-1})
        for m in models:
            est, space = zoo[m]
            # default
            d = clone(est).fit(Xtr, ytr)
            base[f"pred_{m}_def"] = d.predict(Xte); base[f"prob_{m}_def"] = d.predict_proba(Xte)[:, 1]
            # tuned (nested, chronological inner CV)
            t, bp, bs = tune(est, space, Xtr, ytr, n_iter, inner_splits, gap)
            base[f"pred_{m}_tuned"] = t.predict(Xte); base[f"prob_{m}_tuned"] = t.predict_proba(Xte)[:, 1]
            chosen.append(dict(fold=fold, model=m, inner_bacc=bs, params=json.dumps(
                {k: (v if not isinstance(v, (np.generic,)) else v.item()) for k, v in bp.items()}, default=str)))
        preds.append(base); a += test_len; fold += 1
    return pd.concat(preds, ignore_index=True), pd.DataFrame(chosen)

def feature_importance(X, y, init_train):
    rf = RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=10,
                                random_state=SEED, n_jobs=1).fit(X.iloc[:init_train], y.iloc[:init_train])
    return pd.Series(rf.feature_importances_, index=X.columns)

def strategy_stats(df, col, cost_bps=5.0, ann=252):
    pos = np.where(df[col].values == 1, 1.0, -1.0)          # long / short
    turn = np.abs(np.diff(np.r_[0.0, pos])) / 2.0 * 2.0
    gross = pos * df["r"].values
    net = gross - turn * cost_bps * 1e-4
    sh = lambda x: np.sqrt(ann) * x.mean() / x.std(ddof=1) if x.std(ddof=1) > 0 else 0.0
    return dict(ann_ret_net=net.mean() * ann, sharpe_gross=sh(gross), sharpe_net=sh(net))

In [11]:
def main(args=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="results")
    ap.add_argument("--data-dir"); ap.add_argument("--synthetic", action="store_true")
    ap.add_argument("--tickers", default=",".join(UNIVERSE))
    ap.add_argument("--end", default=str(date.today()))
    ap.add_argument("--years", type=float, default=3.0)
    ap.add_argument("--init-train", type=int, default=252)
    ap.add_argument("--test-len", type=int, default=63)
    ap.add_argument("--gap", type=int, default=5)          # purge/embargo (days)
    ap.add_argument("--n-iter", type=int, default=12)
    ap.add_argument("--inner-splits", type=int, default=3)
    ap.add_argument("--models", default="KNN,RF,GBM,ANN")
    ap.add_argument("--leak-control", action="store_true", help="extra run with a deliberately leaky feature")

    # Use an empty list instead of sys.argv when running inside a notebook without arguments
    if args is None:
        args = []
    a = ap.parse_args(args)

    os.makedirs(a.out, exist_ok=True); os.makedirs(os.path.join(a.out, "raw"), exist_ok=True)
    tickers = a.tickers.split(","); models = a.models.split(",")
    end = pd.Timestamp(a.end); start = end - pd.Timedelta(days=int(365.25 * a.years) + 60)  # +60 warm-up for indicators
    if a.synthetic: data = synthetic(tickers)
    elif a.data_dir: data = load_dir(a.data_dir, tickers)
    else: data = download(tickers, start.date(), (end + pd.Timedelta(days=1)).date(), os.path.join(a.out, "raw"))
    if not data: sys.exit("no data loaded")
    all_pred, all_ch, rows, fi, meta = [], [], [], {}, []
    t0 = time.time()
    for t, df in data.items():
        df = df[df.index >= end - pd.Timedelta(days=int(365.25 * a.years) + 60)]
        X, y, r = make_features(df, leak_control=False)
        # evaluation sample = last `years` of target dates only (warm-up rows already consumed by indicators)
        keep = X.index >= end - pd.Timedelta(days=int(365.25 * a.years))
        X, y, r = X[keep], y[keep], r[keep]
        if len(X) < a.init_train + a.test_len + a.gap + 20:
            print("skip", t, len(X)); continue
        pr, ch = walk_forward(X, y, r, a.init_train, a.test_len, a.gap, a.n_iter, a.inner_splits, models)
        pr.insert(0, "asset", t); ch.insert(0, "asset", t)
        all_pred.append(pr); all_ch.append(ch); fi[t] = feature_importance(X, y, a.init_train)
        meta.append(dict(asset=t, cls=UNIVERSE.get(t, "Other"), n_obs=len(X), start=str(X.index[0].date()),
                         end=str(X.index[-1].date()), up_share=float(y.mean()),
                         n_test=len(pr)))
        print(f"{t:10s} obs={len(X):4d} test={len(pr):4d}  elapsed={time.time()-t0:6.0f}s", flush=True)
        if a.leak_control:
            Xl, yl, rl = make_features(df, leak_control=True)
            mk = Xl.index >= X.index[0]; Xl, yl, rl = Xl[mk], yl[mk], rl[mk]
            pl, _ = walk_forward(Xl, yl, rl, a.init_train, a.test_len, a.gap, 4, 2, ["RF"])
            rows.append(dict(asset=t, acc_leaky=accuracy_score(pl["y"], pl["pred_RF_tuned"])))
    P = pd.concat(all_pred, ignore_index=True); C = pd.concat(all_ch, ignore_index=True)
    P.to_csv(os.path.join(a.out, "predictions.csv"), index=False); C.to_csv(os.path.join(a.out, "chosen_params.csv"), index=False)
    pd.DataFrame(fi).to_csv(os.path.join(a.out, "feature_importance.csv"))
    pd.DataFrame(meta).to_csv(os.path.join(a.out, "assets.csv"), index=False)
    if rows: pd.DataFrame(rows).to_csv(os.path.join(a.out, "leak_control.csv"), index=False)
    # per asset x method metrics
    methods = ["AlwaysUp", "Majority", "Persist"] + [f"{m}_{k}" for m in models for k in ("def", "tuned")]
    recs = []
    for t, g in P.groupby("asset"):
        for mth in methods:
            p = g[f"pred_{mth}"].values
            prob = g[f"prob_{mth}"].values if f"prob_{mth}" in g else p.astype(float)
            m = metrics(g["y"].values, p, prob); m.update(strategy_stats(g, f"pred_{mth}"))
            m.update(asset=t, method=mth, n=len(g)); recs.append(m)
    pd.DataFrame(recs).to_csv(os.path.join(a.out, "metrics_by_asset.csv"), index=False)
    json.dump(vars(a), open(os.path.join(a.out, "config.json"), "w"), indent=1)
    print("done in %.0fs" % (time.time() - t0))

if __name__ == "__main__":
    main()

  AAPL         793 rows  2023-08-14 -> 2026-10-09
  MSFT         793 rows  2023-08-14 -> 2026-10-09
  NVDA         793 rows  2023-08-14 -> 2026-10-09
  AMZN         793 rows  2023-08-14 -> 2026-10-09
  JPM          793 rows  2023-08-14 -> 2026-10-09
  XOM          793 rows  2023-08-14 -> 2026-10-09
  JNJ          793 rows  2023-08-14 -> 2026-10-09
  TSLA         793 rows  2023-08-14 -> 2026-10-09
  SPY          793 rows  2023-08-14 -> 2026-10-09
  QQQ          793 rows  2023-08-14 -> 2026-10-09
  IWM          793 rows  2023-08-14 -> 2026-10-09
  EEM          793 rows  2023-08-14 -> 2026-10-09
  ^GDAXI       804 rows  2023-08-14 -> 2026-10-12
  XU100.IS     791 rows  2023-08-14 -> 2026-10-12
  TLT          793 rows  2023-08-14 -> 2026-10-09
  GLD          793 rows  2023-08-14 -> 2026-10-09
  USO          793 rows  2023-08-14 -> 2026-10-09
  EURUSD=X     821 rows  2023-08-14 -> 2026-10-12
  BTC-USD     1156 rows  2023-08-14 -> 2026-10-12
  ETH-USD     1156 rows  2023-08-14 -> 2026-10-12
